# 01 · What is a weight matrix, *really*?

Before we can understand LoRA, we have to understand the one thing LoRA changes:
a **weight matrix**.

We'll build it up from the smallest possible piece — a single neuron — using tiny
numbers you can check on your fingers. **No prior knowledge needed.** Let's go!

In [ ]:
import numpy as np

## Step 1 — A single neuron is just "multiply and add"

Suppose we want to guess a student's **exam score** from two facts about them:

- how many hours they **studied**
- how many hours they **slept**

A *neuron* gives each fact an importance — called a **weight** — then multiplies
and adds. Let's say:

- studying has weight **3** (it matters a lot)
- sleeping has weight **1** (it matters a little)

For a student who studied **2** hours and slept **8** hours:

$$\text{score} = (2 \times 3) + (8 \times 1) = 6 + 8 = 14$$

That multiply-and-add is the **entire** operation a neuron does. Let's check it.

In [ ]:
weights = np.array([3, 1])    # importance of [studying, sleeping]
x       = np.array([2, 8])    # this student: studied 2h, slept 8h

score = np.dot(weights, x)    # (2*3) + (8*1)
print("score =", score)

We get **14**, exactly like our hand calculation.

## Step 2 — Stack two neurons, and the weights become a *matrix*

One neuron predicts one thing. What if we want **two** outputs — say, predict both
the **exam score** *and* the student's **happiness**? We just use **two neurons**,
each with its own weights:

| neuron | weight on studying | weight on sleeping |
|---|---|---|
| exam score | 3 | 1 |
| happiness  | 0 | 2 |

Stacking those weight-rows gives a little **table of numbers** — 2 neurons × 2
inputs. **That table is a weight matrix.** Each *row* is one neuron.

Doing the multiply-and-add for each row:

$$\text{exam} = (2\times 3)+(8\times 1) = 14 \qquad \text{happiness} = (2\times 0)+(8\times 2) = 16$$

In [ ]:
W = np.array([[3, 1],     # row 0 = exam-score neuron
              [0, 2]])    # row 1 = happiness neuron
x = np.array([2, 8])

print("both outputs at once:", W @ x)   # matrix times vector

# the SAME thing, one neuron (one row) at a time:
print("exam  (row 0 · x) =", W[0] @ x)
print("happy (row 1 · x) =", W[1] @ x)

`W @ x` gives `[14, 16]` — and it's identical to dotting each row with `x` by hand.
So a "matrix times a vector" is nothing scary: **it's just several neurons doing
multiply-and-add at the same time.**

## Step 3 — The big picture

A whole layer of a neural network is just:

$$\text{output} = W \, x$$

The matrix **`W` holds all the model's "knobs."** A large language model is millions
of these knobs stacked in many layers. And here is the sentence the rest of this
whole project hangs on:

> **Fine-tuning = nudging the knobs in `W` so the model behaves the way we want.**

Hold onto that. Everything — LoRA included — is a clever idea about *how* to nudge
those knobs.

## Step 4 — One more way to read `W @ x` (we'll need this next time)

We just read the output as *"each **row** dotted with `x`."* There's a second way to
read the exact same calculation, and it'll matter a lot soon: the output is a
**mix of the matrix's columns**, where each input says how much of its column to add.

$$\text{output} = \underbrace{2}_{\text{input 0}} \times \underbrace{\begin{bmatrix}3\\0\end{bmatrix}}_{\text{column 0}} \;+\; \underbrace{8}_{\text{input 1}} \times \underbrace{\begin{bmatrix}1\\2\end{bmatrix}}_{\text{column 1}} = \begin{bmatrix}6\\0\end{bmatrix} + \begin{bmatrix}8\\16\end{bmatrix} = \begin{bmatrix}14\\16\end{bmatrix}$$

Same answer — **14, 16** — just built a different way.

In [ ]:
col0 = W[:, 0]    # [3, 0]
col1 = W[:, 1]    # [1, 2]

print("column view:", x[0] * col0 + x[1] * col1)

Why bother with this second view? Because it shows the **columns are the building-block
directions** that every output is made out of. That raises a very natural question:

> *How many genuinely **different** directions does a matrix have?*

That number has a name — the matrix's **rank** — and it turns out to be the exact key
that makes LoRA possible.

## Recap

- A **neuron** = multiply each input by a weight, then add them up.
- A **layer** = many neurons doing that at once = **`output = W @ x`**.
- `W` is the model's **knobs**; **fine-tuning changes `W`**.
- Every output is built from `W`'s **columns** — and *how many independent columns*
  there are is the next idea: **rank**.

**BAM!** On to **`02 — What is rank?`**